# 2. Inference по базису драйверов

Ноутбук анализирует один выбранный Markdown-кейс, записывает подробный и компактный результаты и формирует requests to add. Каталог во время inference не меняется.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
%cd {ROOT}


/home/elwis/Projects/evaluation-drivers


/home/elwis/Projects/venv/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
from src.config import Settings
from src.file_io import load_catalog
from src.inference import run_inference

# Укажите имя одного файла из data/test (без пути).
CASE_FILENAME = "case_082_optimal_drop_times_using_machine_learning.md"

settings = Settings.from_env()
catalog = load_catalog(settings.driver_catalog_path)
case_path = settings.test_cases_dir / CASE_FILENAME
if not case_path.is_file():
    raise FileNotFoundError(f"Test case does not exist: {case_path}")
print(f"Test case: {case_path}")
print(f"Catalog v{catalog.catalog_version}: {len(catalog.drivers)} drivers")
print(f"Model: {settings.openai_model}")


Test case: data/test/case_082_optimal_drop_times_using_machine_learning.md
Catalog v16: 48 drivers
Model: gpt-4.1-mini


In [3]:
results = run_inference(settings, CASE_FILENAME)
result = results[0]
print(f"Processed: {result['case_id']}")


Processed: case_082


In [4]:
import pandas as pd
compact_path = settings.artifacts_dir / "inference" / "compact_results.csv"
pd.read_csv(compact_path)


,case_id,driver_id,relevance,evidence_status,top_category,top_probability,distribution
0,case_082,business.scope_maturity,relevant,explicit,well_defined,0.9,"{""exploratory"": 0.0, ""partial"": 0.1, ""unknown""..."
1,case_082,data.quality,relevant,explicit,high,0.7,"{""critical"": 0.0, ""high"": 0.7, ""low"": 0.0, ""mo..."
2,case_082,data.source_count,insufficient_context,not_stated,unknown,1.0,"{""unknown"": 1.0}"
3,case_082,data.source_heterogeneity,insufficient_context,not_stated,unknown,1.0,"{""unknown"": 1.0}"
4,case_082,data.scale,insufficient_context,not_stated,unknown,1.0,"{""unknown"": 1.0}"
5,case_082,data.sensitivity,insufficient_context,not_stated,unknown,1.0,"{""unknown"": 1.0}"
6,case_082,evaluation.human_in_loop,insufficient_context,not_stated,unknown,1.0,"{""unknown"": 1.0}"
7,case_082,evaluation.target_quality,relevant,explicit,high,0.8,"{""business_acceptable"": 0.15, ""exploratory"": 0..."
8,case_082,experimentation.online_validation,relevant,explicit,pilot,0.9,"{""ab_test"": 0.0, ""multi_stage"": 0.0, ""none"": 0..."
9,case_082,functional.use_case_count,insufficient_context,not_stated,unknown,1.0,"{""unknown"": 1.0}"


## Полнота информации и приоритет уточнений

Полнота — взвешенная доля известной информации по применимым драйверам. Весом служит `cost_impact_percent`, а долей неизвестности — вероятность категории `unknown`. Уточнение одного драйвера снимает его взвешенную неизвестность; прирост ниже выражен в процентных пунктах общей полноты.


In [5]:
from src.information_coverage import calculate_information_coverage
from src.schemas import DriverInference

driver_results = [DriverInference.model_validate(item) for item in result["driver_results"]]
coverage, clarification_priorities = calculate_information_coverage(driver_results, catalog)
print(f"Суммарная заполненность информации по базису: {coverage:.1f}%")


Суммарная заполненность информации по базису: 27.4%


In [6]:
priority_table = pd.DataFrame(clarification_priorities).rename(columns={
    "driver_id": "Драйвер",
    "driver_name": "Название",
    "cost_impact_percent": "Влияние на стоимость, %",
    "unknown_probability_percent": "Неизвестность, %",
    "coverage_gain_percentage_points": "Прирост полноты, п.п.",
    "removable_uncertainty_percent": "Снимаемая неопределённость, %",
})
priority_table = priority_table[priority_table["Прирост полноты, п.п."] > 0].reset_index(drop=True)
priority_table.index += 1
priority_table.round(2)


,Драйвер,Название,"Влияние на стоимость, %","Неизвестность, %","Прирост полноты, п.п.","Снимаемая неопределённость, %"
1,functional.use_case_count,Количество независимых use cases,35.0,100.0,5.37,7.39
2,integration.system_count,Количество системных интеграций,30.0,100.0,4.60,6.34
3,model.training_stability,Стабильность процесса обучения модели,30.0,100.0,4.60,6.34
4,data.scale,Масштаб обрабатываемых данных,25.0,100.0,3.83,5.28
5,model.modality_count,Количество модальностей,25.0,100.0,3.83,5.28
6,performance.latency_requirement,Строгость latency-требований,25.0,100.0,3.83,5.28
7,reliability.availability,Требования к доступности,25.0,100.0,3.83,5.28
8,scalability.load_level,Уровень нагрузки,25.0,100.0,3.83,5.28
9,data.source_heterogeneity,Неоднородность источников,22.0,100.0,3.37,4.65
10,data.sensitivity,Чувствительность данных,20.0,100.0,3.07,4.22
